# ⚡ NeuralForge Quickstart

Train a model, export to ONNX, and visualize — all in one notebook.

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
from neural_forge.core import NeuralForgeEngine, ForgeConfig
from neural_forge.training import Trainer, TrainConfig
from neural_forge.onnx_utils import ONNXModelZoo

# Init engine (auto-detects RTX 5090)
engine = NeuralForgeEngine(ForgeConfig(precision='bf16', compile_model=True))
print(engine.info())

In [ ]:
# Quick CNN model
import torch.nn as nn

class QuickCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.net(x)

model = QuickCNN(num_classes=10)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Train on CIFAR-10
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader

transform = T.Compose([T.ToTensor(), T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
train_ds = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=transform)
val_ds   = torchvision.datasets.CIFAR10('./data', train=False, transform=transform)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=8, pin_memory=True)

trainer = Trainer(
    model, 
    TrainConfig(epochs=5, lr=1e-3, precision='bf16', compile_model=True),
    engine.device
)
metrics = trainer.fit(train_loader, val_loader)
print(f'\nFinal val acc: {metrics.val_acc*100:.2f}%')

In [ ]:
# Export to ONNX
zoo = ONNXModelZoo()
dummy = torch.randn(1, 3, 32, 32).to(engine.device)
path = zoo.export_pytorch(model, dummy, name='quickcnn_cifar10')
print(f'Exported: {path}')
print('Open Netron to visualize: https://netron.app')

In [ ]:
# Browse ONNX Model Zoo
models = zoo.list_models(category='vision/classification')
for m in models:
    cached = '✓' if m['cached'] else ' '
    print(f'[{cached}] {m["name"]:20} | {m["description"]}')